# Semi-supervised learning: a hands-on implementation

**Track:** Machine learning · **Guide:** [`012-semi-supervised-learning.md`](../guides/concepts/012-semi-supervised-learning.md)

Semi-supervised learning combines a small labeled set with a larger unlabeled set to improve learning when labels are scarce.

> **Intuition:** A teacher labels a few examples; the learner uses them to organize a much larger unlabeled pile.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement Semi-supervised learning]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** Use pseudo-labels for document triage only after estimating error and keeping uncertain examples for human review.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Separate labeled and unlabeled examples

The unlabeled examples have features but no target. The `-1` marker is the scikit-learn convention for unknown labels.


In [ ]:
import numpy as np

X = np.array([[0.0], [0.2], [0.4], [4.8], [5.0], [5.2]])
y = np.array([0, -1, -1, -1, -1, 1])
print("labeled:", int((y >= 0).sum()), "unlabeled:", int((y < 0).sum()))

## 2. Propagate labels from nearby points

Label propagation assumes nearby examples tend to share labels. Wrong labels can spread, so the assumption needs testing.


In [ ]:
from sklearn.semi_supervised import LabelSpreading

model = LabelSpreading(kernel="knn", n_neighbors=2).fit(X, y)
print("inferred labels:", model.transduction_.tolist())

## 3. Keep uncertainty visible

Only use high-confidence pseudo-labels for a downstream task; route uncertain cases for annotation.


In [ ]:
confidence = model.label_distributions_.max(axis=1)
for index, (label, score) in enumerate(zip(model.transduction_, confidence)):
    status = "pseudo-labeled" if score >= 0.8 else "needs review"
    print(index, int(label), round(float(score), 2), status)

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/012-semi-supervised-learning.md) and the
[learning roadmap](../guides/ROADMAP.md).
